## Scenario 1: A single data scientist participating in an ML competition

This scenario demonstrates how an individual data scientist can use MLflow to track machine learning experiments on their local machine. This is a common setup for solo projects, hackathons, or competitions where collaboration and remote access are not required.

### MLflow setup overview
- **Tracking server:** Not used (runs locally, no remote server)
- **Backend store:** Local filesystem (stores experiment metadata in the `mlruns/` folder)
- **Artifacts store:** Local filesystem (stores model files and other artifacts in the same `mlruns/` folder)

With this setup, all experiment runs, parameters, metrics, and artifacts are saved locally. You can explore and compare your experiments using the MLflow UI.

### How to use the MLflow UI
- You can launch the MLflow UI by running `mlflow ui` in your terminal, or by running the provided code cell in this notebook.
- The UI will be available at [http://localhost:5000](http://localhost:5000).
- Use the UI to browse experiments, compare runs, and inspect logged models and artifacts.

> **Tip:** This local setup is ideal for learning and prototyping. For team projects or production, you would typically use a remote tracking server and a more robust backend (e.g., a database and cloud storage).

In [1]:
%pip install mlflow
import mlflow

  Using cached mlflow-3.10.0-py3-none-any.whl.metadata (31 kB)
  Using cached mlflow_skinny-3.10.0-py3-none-any.whl.metadata (32 kB)
  Using cached mlflow_tracing-3.10.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cachetools-7.0.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached databricks_sdk-0.96.0-py3-none-any.whl.metadata (40 kB)
  Using cache

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'sqlite:///C:/Users/nuria/OneDrive/Documentos/MBDS%202/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlflow.db'


In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location=('file:///c:/Users/nuria/OneDrive/Documentos/MBDS '
  '2/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/1'), creation_time=1772554530792, experiment_id='1', last_update_time=1772554530792, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location=('file:///c:/Users/nuria/OneDrive/Documentos/MBDS '
  '2/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/0'), creation_time=1772554512139, experiment_id='0', last_update_time=1772554512139, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

### Creating an experiment and logging a new run

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import mlflow
import sklearn
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_experiment("my-experiment-1")

X, y = load_iris(return_X_y=True)
class_names = load_iris().target_names

models = [
    ("LogisticRegression", LogisticRegression(C=1.0, random_state=42, max_iter=1000, solver="lbfgs")),
    ("RandomForestClassifier", RandomForestClassifier(n_estimators=100, random_state=42))
]

for model_name, model in models:
    with mlflow.start_run() as run:
        mlflow.set_tag("model_type", model_name)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        model.fit(X, y)
        y_pred = model.predict(X)
        acc = accuracy_score(y, y_pred)
        mlflow.log_metric("accuracy", acc)
        # Log confusion matrix as labeled DataFrame
        cm = confusion_matrix(y, y_pred)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        cm_df.to_csv("confusion_matrix_labeled.csv")
        mlflow.log_artifact("confusion_matrix_labeled.csv")
        # Confusion matrix heatmap as image
        plt.figure(figsize=(5,4))
        sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix ({model_name})")
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig("confusion_matrix_heatmap.png")
        plt.close()
        mlflow.log_artifact("confusion_matrix_heatmap.png")
        # Provide input_example and use 'name' instead of deprecated 'artifact_path'
        input_example = np.expand_dims(X[0], axis=0)
        mlflow.sklearn.log_model(model, name="models", input_example=input_example)
        mlflow.set_tag("n_classes", len(np.unique(y)))
        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("description", f"{model_name} on Iris dataset")
        print(f"Logged run for {model_name}, accuracy={acc:.3f}")

2026/03/03 18:13:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run for LogisticRegression, accuracy=0.973


2026/03/03 18:14:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run for RandomForestClassifier, accuracy=1.000


In [12]:
mlflow.search_experiments()

[<Experiment: artifact_location=('file:///c:/Users/nuria/OneDrive/Documentos/MBDS '
  '2/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/1'), creation_time=1772554530792, experiment_id='1', last_update_time=1772554530792, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location=('file:///c:/Users/nuria/OneDrive/Documentos/MBDS '
  '2/MLOPS/ie-mlops-nyc-taxis/03-experiment-tracking/mlruns/0'), creation_time=1772554512139, experiment_id='0', last_update_time=1772554512139, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [13]:
# Launch MLflow UI (default: uses local mlruns/ folder)
import subprocess
import sys

# This will run 'mlflow ui' as a background process
subprocess.Popen([sys.executable, '-m', 'mlflow', 'ui'])
print("MLflow UI started. Open http://localhost:5000 in your browser.")

MLflow UI started. Open http://localhost:5000 in your browser.


### Interacting with the model registry

In [14]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [15]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException:
    print("It's not possible to access the model registry :(")